In [17]:
!pip install openai sounddevice numpy scipy jiwer

In [18]:
!pip install python-dotenv
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("✅ API Key geladen" if api_key else "❌ Kein Key gefunden – .env prüfen")

✅ API Key geladen


In [22]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile

DURATION = 45
SAMPLE_RATE = 16000

print("🔴 Aufnahme läuft – jetzt sprechen!")
audio = sd.rec(int(DURATION * SAMPLE_RATE), 
               samplerate=SAMPLE_RATE, 
               channels=1, 
               dtype='float32')
print(f"⏱️ Läuft {DURATION} Sekunden... Nächste Zelle ausführen zum Stoppen.")

🔴 Aufnahme läuft – jetzt sprechen!
⏱️ Läuft 45 Sekunden... Nächste Zelle ausführen zum Stoppen.


In [23]:
sd.stop()

audio_clipped = np.clip(audio, -1, 1)
audio_int16 = (audio_clipped * 32767).astype(np.int16)
wavfile.write('my_audio_recording.wav', SAMPLE_RATE, audio_int16)
print("✅ Gespeichert als 'my_audio_recording.wav'")

✅ Gespeichert als 'my_audio_recording.wav'


In [24]:
from IPython.display import Audio
Audio('my_audio_recording.wav')

In [25]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

with open("my_audio_recording.wav", "rb") as audio_file:
    whisper_result = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json"
    )

print("📝 Whisper Transkription:")
print(whisper_result.text)
print(f"\n⏱️ Dauer: {whisper_result.duration:.1f} Sekunden")

📝 Whisper Transkription:
Hallo, ich erzähle euch jetzt einfach mal von meinem Morgen. Ich war früh wach und bin dann schon spazieren gegangen. Ich habe dort auf dem Weg einen alten Bekannten getroffen aus meiner Heimatstadt und dann haben wir uns sehr gut unterhalten. Es war schön einfach mal wieder ein Update zu bekommen, wie es ihm auch geht. Und dann bin ich nach Hause gekommen und dann ging auch schon das Bootcamp los.

⏱️ Dauer: 45.0 Sekunden


In [44]:
print(whisper_result.text)

Hallo, ich erzähle euch jetzt einfach mal von meinem Morgen. Ich war früh wach und bin dann schon spazieren gegangen. Ich habe dort auf dem Weg einen alten Bekannten getroffen aus meiner Heimatstadt und dann haben wir uns sehr gut unterhalten. Es war schön einfach mal wieder ein Update zu bekommen, wie es ihm auch geht. Und dann bin ich nach Hause gekommen und dann ging auch schon das Bootcamp los.


In [45]:
from IPython.display import Audio
Audio('my_audio_recording.wav')

In [60]:
# 1. Aufnahme in Zelle 4 nochmal anhören
# 2. Whisper-Output oben als Basis nehmen
# 3. Fehler manuell korrigieren und hier eintragen

ground_truth_text = "Hallo, ich erzähle euch jetzt einfach mal von meinem Morgen. Ich war früh wach und bin dann schon spazieren gegangen. Ich hab dort auf dem Weg einen alten Bekannten getroffen aus meiner Heimatstadt und dann haben wir uns sehr gut unterhalten. Es war schön einfach mal wieder ein Update zu bekommen, wie`s ihm auch geht und dann bin ich nach Hause gekommen und dann ging auch schon das Bootcamp los."
print(f"✅ Ground Truth gesetzt – {len(ground_truth_text.split())} Wörter")

✅ Ground Truth gesetzt – 70 Wörter


In [61]:
%%writefile wer_calc.py
import jiwer
import json

def calculate_wer(reference, hypothesis):
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])
    ref_clean = transformation(reference)
    hyp_clean = transformation(hypothesis)
    output = jiwer.process_words(ref_clean, hyp_clean)
    results = {
        "wer": output.wer,
        "accuracy": 1 - output.wer,
        "substitutions": output.substitutions,
        "insertions": output.insertions,
        "deletions": output.deletions,
        "hits": output.hits,
        "reference_words": len(ref_clean.split()),
        "hypothesis_words": len(hyp_clean.split())
    }
    return results

Overwriting wer_calc.py


In [62]:
from wer_calc import calculate_wer

wer_results = calculate_wer(ground_truth_text, whisper_result.text)
print(wer_results)

with open("wer_results.json", "w") as f:
    json.dump(wer_results, f, indent=2)
print("gespeichert")

{'wer': 0.04285714285714286, 'accuracy': 0.9571428571428572, 'substitutions': 2, 'insertions': 1, 'deletions': 0, 'hits': 68, 'reference_words': 70, 'hypothesis_words': 71}
gespeichert


In [63]:
PRICE_PER_MINUTE = 0.006

audio_duration_sec = whisper_result.duration
audio_duration_min = audio_duration_sec / 60
cost_single = audio_duration_min * PRICE_PER_MINUTE

scenarios = {
    "1 Aufnahme (dein Test)": audio_duration_min,
    "100 Meetings/Monat (1h je)": 100 * 60,
    "500 Meetings/Monat (1h je)": 500 * 60,
}

for label, minutes in scenarios.items():
    print(label + ": $" + str(round(minutes * PRICE_PER_MINUTE, 4)))

cost_data = {
    "price_per_minute_usd": PRICE_PER_MINUTE,
    "audio_duration_seconds": audio_duration_sec,
    "cost_this_recording_usd": cost_single,
    "scenarios": {k: round(v * PRICE_PER_MINUTE, 4) for k, v in scenarios.items()}
}

with open("cost_analysis.json", "w") as f:
    json.dump(cost_data, f, indent=2)
print("gespeichert")

1 Aufnahme (dein Test): $0.0045
100 Meetings/Monat (1h je): $36.0
500 Meetings/Monat (1h je): $180.0
gespeichert


In [64]:
%%writefile generate_report.py
def generate_report(wer_results, audio_duration_sec, price_per_minute):
    audio_duration_min = audio_duration_sec / 60
    cost_single = audio_duration_min * price_per_minute

    lines = [
        "WHISPER STT EVALUATION REPORT",
        "==============================",
        "",
        "1. ACCURACY ANALYSIS",
        "WER:             " + str(round(wer_results["wer"]*100, 2)) + "%",
        "Accuracy:        " + str(round(wer_results["accuracy"]*100, 2)) + "%",
        "Substitutions:   " + str(wer_results["substitutions"]),
        "Insertions:      " + str(wer_results["insertions"]),
        "Deletions:       " + str(wer_results["deletions"]),
        "Korrekte Woerter: " + str(wer_results["hits"]) + " / " + str(wer_results["reference_words"]),
        "",
        "2. COST ANALYSIS",
        "Audio-Dauer:          " + str(round(audio_duration_sec, 1)) + " sec",
        "Kosten diese Datei:   $" + str(round(cost_single, 5)),
        "100 Meetings (1h/je): $" + str(round(100 * 60 * price_per_minute, 2)) + "/Monat",
        "500 Meetings (1h/je): $" + str(round(500 * 60 * price_per_minute, 2)) + "/Monat",
        "",
        "3. PERFORMANCE INSIGHTS",
        "[Eigene Beobachtungen: Fehlertypen, Fachbegriffe, Pausen, Akzent]",
        "",
        "4. RECOMMENDATIONS",
        "[Deploy Ja/Nein - unter welchen Bedingungen?]",
    ]
    return "\n".join(lines)


Overwriting generate_report.py


In [65]:
from generate_report import generate_report

report = generate_report(wer_results, audio_duration_sec, PRICE_PER_MINUTE)
print(report)

with open("evaluation_report.txt", "w") as f:
    f.write(report)
print("gespeichert")

WHISPER STT EVALUATION REPORT

1. ACCURACY ANALYSIS
WER:             4.29%
Accuracy:        95.71%
Substitutions:   2
Insertions:      1
Deletions:       0
Korrekte Woerter: 68 / 70

2. COST ANALYSIS
Audio-Dauer:          45.0 sec
Kosten diese Datei:   $0.0045
100 Meetings (1h/je): $36.0/Monat
500 Meetings (1h/je): $180.0/Monat

3. PERFORMANCE INSIGHTS
[Eigene Beobachtungen: Fehlertypen, Fachbegriffe, Pausen, Akzent]

4. RECOMMENDATIONS
[Deploy Ja/Nein - unter welchen Bedingungen?]
gespeichert
